In [1]:
import pandas as pd
import numpy as np

ledger = pd.read_csv("ledger.csv")
gateway = pd.read_csv("gateway.csv")

In [2]:
print("Ledger Nulls:\n", ledger.isnull().sum())
print("Gateway Nulls:\n", gateway.isnull().sum())

print("Ledger Duplicates:", ledger.duplicated().sum())
print("Gateway Duplicates:", gateway.duplicated().sum())

Ledger Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Ledger Duplicates: 0
Gateway Duplicates: 0


In [5]:
key = "transaction_id"
missing_in_gateway = ledger[~ledger[key].isin(gateway[key])]
missing_in_gateway.to_csv("missing_in_gateway.csv", index=False)
missing_in_ledger = gateway[~gateway[key].isin(ledger[key])]
missing_in_ledger.to_csv("missing_in_ledger.csv", index=False)

In [6]:
merged = pd.merge(
    ledger,
    gateway,
    on=key,
    how="inner",
    suffixes=("_ledger", "_gateway")
)

In [8]:
amount_mismatches = merged[
    merged["amount_usd_ledger"] != merged["amount_usd_gateway"]
]

amount_mismatches.to_csv("amount_mismatches.csv", index=False)

In [9]:
status_mismatches = merged[
    merged["status_ledger"] != merged["status_gateway"]
]

status_mismatches.to_csv("status_mismatches.csv", index=False)

In [11]:
merged["amount_match"] = merged["amount_usd_ledger"] == merged["amount_usd_gateway"]
merged["status_match"] = merged["status_ledger"] == merged["status_gateway"]

merged["reconciliation_status"] = np.where(
    (merged["amount_match"]) & (merged["status_match"]),
    "matched",
    "mismatched"
)

merged.to_csv("reconciliation_report.csv", index=False)

In [12]:
summary = {
    "total_ledger_records": len(ledger),
    "total_gateway_records": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches),
    "fully_matched": len(merged[
        (merged["amount_match"]) & (merged["status_match"])
    ])
}

import json
with open("summary_metrics.json", "w") as f:
    json.dump(summary, f, indent=4)

print(summary)

{'total_ledger_records': 10, 'total_gateway_records': 9, 'missing_in_gateway': 2, 'missing_in_ledger': 1, 'amount_mismatches': 2, 'status_mismatches': 1, 'fully_matched': 5}


In [13]:
import pandas as pd
import json

with open("api_response_sample.json") as f:
    data = json.load(f)

In [14]:
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['generated_at', 'source', 'batches'])


In [17]:
# Fully flatten the JSON, including the 'settlements' list
df_normalized = pd.json_normalize(
    data['batches'],
    record_path=['settlements'],
    meta=['batch_id', ['merchant', 'merchant_id'], ['merchant', 'merchant_name'], ['merchant', 'region']]
)

In [18]:
# Clean column names: replace '.' with '_' and convert to lowercase
df_normalized.columns = [col.replace('.', '_').lower() for col in df_normalized.columns]

# Display the cleaned column names
print("Cleaned Column Names:", df_normalized.columns.tolist())

Cleaned Column Names: ['settlement_id', 'amount_usd', 'status', 'processed_at', 'bank_name', 'bank_country', 'batch_id', 'merchant_merchant_id', 'merchant_merchant_name', 'merchant_region']


In [19]:
# Convert date/time fields
for col in df_normalized.columns:
    if 'date' in col or 'at' in col:
        df_normalized[col] = pd.to_datetime(df_normalized[col], errors='coerce')

# Display data types to confirm conversion
print("\nDataFrame Info after Date Conversion:")
df_normalized.info()


DataFrame Info after Date Conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   settlement_id           6 non-null      object             
 1   amount_usd              6 non-null      float64            
 2   status                  0 non-null      datetime64[ns]     
 3   processed_at            6 non-null      datetime64[ns, UTC]
 4   bank_name               6 non-null      object             
 5   bank_country            6 non-null      object             
 6   batch_id                0 non-null      datetime64[ns]     
 7   merchant_merchant_id    6 non-null      object             
 8   merchant_merchant_name  6 non-null      object             
 9   merchant_region         6 non-null      object             
dtypes: datetime64[ns, UTC](1), datetime64[ns](2), float64(1), object(6)
memory 

/tmp/ipykernel_10199/4093286272.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_normalized[col] = pd.to_datetime(df_normalized[col], errors='coerce')
/tmp/ipykernel_10199/4093286272.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_normalized[col] = pd.to_datetime(df_normalized[col], errors='coerce')


In [21]:
# Save the normalized output to a CSV file
df_normalized.to_csv('api_normalized.csv', index=False)

print("\nNormalized data saved to 'api_normalized.csv'")

# Display the first few rows of the final normalized DataFrame
display(df_normalized.head())


Normalized data saved to 'api_normalized.csv'


,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,batch_id,merchant_merchant_id,merchant_merchant_name,merchant_region
0,S001,1520.5,NaT,2026-03-07 08:10:00+00:00,Bank A,IN,NaT,M001,Alpha Mart,APAC
1,S002,980.0,NaT,2026-03-07 08:45:00+00:00,Bank A,IN,NaT,M001,Alpha Mart,APAC
2,S003,640.0,NaT,2026-03-07 09:15:00+00:00,Bank B,SG,NaT,M001,Alpha Mart,APAC
3,S004,2100.0,NaT,2026-03-07 08:20:00+00:00,Bank C,US,NaT,M004,Delta Travels,US
4,S005,500.0,NaT,2026-03-07 08:50:00+00:00,Bank C,US,NaT,M004,Delta Travels,US
